# Deliverable 2 – INF201: Campus Coffee Shop Sales Pipeline

**Gruppe:** [Sander Sebastian Vikse], [Amalie Hestad Berntsen]
**GitHub-repo:** [https://github.com/sirjimcode/INF201_deliverable2]

I denne innleveringen bygger vi en pipeline som finner loggfiler fra kassaapparatene, renser transaksjonsdataene og lager oppsummerende rapporter. Funksjonene lages i oppgave 1–4 og kobles sammen til en full arbeidsflyt i oppgave 5.

In [25]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

Task1

In [26]:
# 1. Write a robust file reader function:


def read_log_file(file_path):

    """
   Funksjonen prøver først å lese filen med UTF-8-koding. Hvis det mislykkes, prøver den ISO-8859-1-koding.
   Den returnerer også en tuple.
    """
    path_obj = Path(file_path)

    try:
        content = path_obj.read_text(encoding='utf-8')
        encoding_used = 'utf-8'
    except UnicodeDecodeError:
        content = path_obj.read_text(encoding='iso-8859-1')
        encoding_used = 'iso-8859-1'

    return (content, encoding_used)


# 2. Write a log parser function that uses regular expressions.
# Log files contain valid sales lines mixed with system noise.
def parse_log_lines(log_innhold):
    """
    Splitter linjene, deretter itererer hver linje for å finne de 6 nøkleverdiene: timestamp,
    register_id, item, price, currency og status. Returnerer en liste med ordbøker som representerer hver transaksjon
    i mønstret som er definert i regexen. Hvis en linje ikke samsvarer med mønsteret, ignoreres den.
    """
    pattern = (
        r"^\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
        r"\[(.*?)\] "
        r"item=(.*?) "
        r"price=([\d.]+) "
        r"([A-Z]{3}) "
        r"\((SUCCESS|REFUND|CANCELLED)\)$"
    )
    transactions = []
    for line in log_innhold.splitlines():
        line = line.strip()
        match = re.search(pattern, line)
        if match:
            record = {
                "timestamp": match.group(1),
                "register_id": match.group(2),
                "item": match.group(3),
                "price": float(match.group(4)),
                "currency": match.group(5),
                "status": match.group(6)
            }
            transactions.append(record)
    return transactions


Vi prøver UTF-8 før Latin-1 fordi UTF-8 feiler tydelig på Latin-1-tegn, mens Latin-1 aldri feiler og ville lest UTF-8-filer med feil tegn uten å si fra, og regex-mønsteret sørger for at bare linjer med alle seks feltene i riktig format blir med.

Task 2

In [27]:
def find_log_files(root_dir):
    """
    Søker gjennom rotmappen med en løkke og rekursiv gjenkalling og finner alle .log-filer, inkludert de i undermapper.
    Returnerer en liste med Path-objekter for hver funnet loggfil.
    """
    root_path = Path(root_dir)
    log_files = []

    # Sjekk at mappen faktisk finnes før vi starter
    if not root_path.exists() or not root_path.is_dir():
        return log_files

    for item in root_path.iterdir():
        if item.is_file() and item.suffix == ".log":
            # Fant en loggfil: legg til i listen
            log_files.append(item)
        elif item.is_dir():
            # Fant en undermappe: kall funksjonen rekursivt og slå sammen resultatene
            log_files.extend(find_log_files(item))

    return log_files


def find_active_registers(data) -> set[str]:
    """
    En funksjon som leter gjennom en JSON-struktur (som kan være en ordbok eller en liste) og finner alle aktive kasser.
    Returnerer et sett med ID-er for aktive kasser.
    """
    active_ids = set()

    # Tilfelle 1: Vi står i en dictionary
    if isinstance(data, dict):
        # Sjekker om denne ordboken representerer en aktiv kasse
        if "id" in data and data.get("active") is True:
            active_ids.add(str(data["id"]))

        # Traverserer videre nedover ved alle verdier i ordboken
        for value in data.values():
            active_ids.update(find_active_registers(value))

    # Tilfelle 2: Vi står i en liste
    elif isinstance(data, list):
        # Traverserer hvert element i listen rekursivt
        for item in data:
            active_ids.update(find_active_registers(item))

    # Tilfelle 3 (Base case): primitive typer som int, str, bool
    # Gjør ingenting, dermed returnerer den bare det oppsamlede settet (eller et tomt sett)

    return active_ids


Begge funksjonene bruker rekursjon der hvert kall har sin egen lokale liste eller sett som returneres og slås sammen i kallet over, og ved å håndtere dict, liste og andre typer hver for seg fungerer uansett hvordan JSON-filen er bygget opp.

Task 3

In [28]:
def remove_price_outliers(price_data) -> np.ndarray:
    """
    Først konverterer input til en 1D NumPy float array. Deretter identifiserer og
    fjerner outliers basert på 3-sigma regelen. Outliers erstattes med medianen av de gyldige prisene.
    Returnerer en ny NumPy array med rensede priser.
    """
    # 1. Konverterer til 1D NumPy float array
    prices = np.array(price_data, dtype=float).flatten()

    # Hvis arrayet er tomt, så returnerer det direkte
    if prices.size == 0:
        return prices

    # 2. Beregner gjennomsnittet (mu) og standardavviket (sigma)
    mu = np.mean(prices)
    sigma = np.std(prices)

    # 3. Boolsk maske for å identifisere outliers, dette betyr verdier over tre standardavvik unna
    outlier_mask = np.abs(prices - mu) > (3 * sigma)

    # 4. Her finner vi gyldige priser
    valid_prices = prices[~outlier_mask]

    # Hvis alle eller ingen verdier er outliers, trengs ingen erstatning da kjører vi bare returnerer original arrayet
    if valid_prices.size == 0 or not np.any(outlier_mask):
        return prices

    # 5. Beregner medianen av de gyldige prisene
    valid_median = np.median(valid_prices)

    # 6. Kloner arrayet og erstatter outliers med medianen
    cleaned_prices = prices.copy()
    cleaned_prices[outlier_mask] = valid_median

    return cleaned_prices


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """
    Beregner forholdet mellom refusjoner og salg. For å unngå deling på null, legger vi til en liten epsilon-verdi.
    Returnerer en NumPy array med forholdet.
    """
    epsilon = 1e-6
    return refund_count / (sales_count + epsilon)


Vi erstatter outliers med medianen fordi den ikke påvirkes av ekstreme verdier, men 3σ-regelen krever nok datapunkter (minst 11) for at en outlier i det hele tatt skal kunne oppdages, noe som ikke er et problem med vårt datasett.

Task 4

In [29]:
def create_transaction_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """
    Tar inn en liste med dictionaries som representerer transaksjoner og konverterer den til en Pandas DataFrame.
    """
    # 1. Konverterer listen med dictionaries til en DataFrame
    df = pd.DataFrame(transactions)

    # 2. Gjør om timestamp til datetime objekter
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # 3. Setter timestamp som indeksen til tabellen
    df = df.set_index("timestamp")

    # 4. Sorter DataFramen kronologisk etter tidsstempel-indeksen
    df = df.sort_index()

    return df


def resample_hourly_item_revenue(df: pd.DataFrame) -> pd.Series:
    """
    Her tar funksjonen inn en DataFrame med transaksjoner og resampler
    den for å beregne total inntekt per vare per time.
    """
    return df.groupby('item').resample('1h')['price'].sum()


def create_status_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Lager en Pivot-tabell som viser total inntekt per vare, gruppert etter status (SUCCESS, REFUND, CANCELLED).
    """
    pivot_df = df.pivot_table(
        values="price",
        index="item",
        columns="status",
        aggfunc="sum",
        fill_value=0.0,
    )

    return pivot_df


def calculate_item_refund_ratio(df: pd.DataFrame) -> pd.Series:
    """
    Beregner forholdet mellom antall refusjoner og antall salg for hver vare.
    Returnerer en Pandas Series med varenavn som indeks og forholdet som verdier.
    """
    # 1. Teller opp antall transaksjoner per vare og status
    counts = pd.crosstab(df["item"], df["status"])

    # 2. Henter ut antall vellykkede salg SUCCESS som et NumPy-array
    if "SUCCESS" in counts.columns:
        sales_count = counts["SUCCESS"].to_numpy()
    else:
        sales_count = np.zeros(len(counts))

    # 3. Henter ut antall refusjoner REFUND som et NumPy-array
    if "REFUND" in counts.columns:
        refund_count = counts["REFUND"].to_numpy()
    else:
        refund_count = np.zeros(len(counts))

    # 4. Kaller funksjonen fra Task 3
    ratios = compute_refund_ratio(sales_count, refund_count)

    # 5. Returnerer som en Pandas Series koblet til varenavnene
    return pd.Series(ratios, index=counts.index, name="refund_ratio")


Timestamp må være datetime-indeks for at resample skal fungere, og i refusjonsforholdet har vi tolket sales count som antall SUCCESS-transaksjoner, mens pivottabellen gjør det lett å sammenligne omsetning både mellom varer og mellom statuser. Man kan tenke på det som data over to akser og ikke en. ALtså krysspunktet. 

Task 5 

In [30]:
def run_full_workflow(
    logs_dir: str | Path = "data/2026",
    registers_json_path: str | Path | None = "data/store_config.json",
    output_dir: str | Path = "output",
) -> None:
    """Kjører hele arbeidsflyten fra rålogg til ferdige rapporter (Steg 1-7)."""
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    # 1. Finner alle loggfiler i den angitte mappen (inkludert undermapper) og leser JSON-registerdata hvis gitt
    log_files = find_log_files(logs_dir)
    if not log_files:
        print("Ingen loggfiler funnet i den angitte mappen.")
        return

    active_registers: set[str] = set()
    if registers_json_path is not None and Path(registers_json_path).exists():
        with open(registers_json_path, encoding="utf-8") as json_file:
            json_data = json.load(json_file)
            active_registers = find_active_registers(json_data)

    # 2. Leser og parser hver loggfil, samler alle transaksjoner i en liste av dictionaries
    parsed_records: list[dict] = []
    for file_path in log_files:
        content, _ = read_log_file(file_path)
        records = parse_log_lines(content)
        parsed_records.extend(records)

    # 3. Bygger en Pandas DataFrame fra listen av dictionaries
    df = create_transaction_dataframe(parsed_records)

    # Filtrerer til kun aktive kasser hvis registerdata ble funnet
    if active_registers:
        df = df[df["register_id"].isin(active_registers)].copy()

    # 4. Bruker remove_price_outliers for å rense prisene
    df["price"] = remove_price_outliers(df["price"].to_numpy())

    # 5. konverterer NOK-priser til EUR ved å multiplisere med 10.0
    nok_mask = df["currency"] == "NOK"
    df.loc[nok_mask, "price"] *= 10.0

    # 6. Generererer rapporter: timebasert inntekt, status pivot-tabell og refusjonsforhold
    hourly_revenue = resample_hourly_item_revenue(df)
    status_pivot = create_status_pivot_table(df)
    refund_ratios = calculate_item_refund_ratio(df)

    # 7. Lagrer alle rapportene til CSV- og JSON-filer i output-mappen
    if active_registers:
        with open(output_path / "active_registers.txt", "w", encoding="utf-8") as f:
            for reg_id in sorted(active_registers):
                f.write(f"{reg_id}\n")

    df.to_csv(output_path / "cleaned_transactions.csv")
    hourly_revenue.reset_index().to_json(
        output_path / "hourly_revenue.json",
        orient="records",
        date_format="iso",
        indent=2,
    )
    status_pivot.to_csv(output_path / "status_pivot_table.csv")
    refund_ratios.to_csv(output_path / "item_refund_ratios.csv")


if __name__ == "__main__":
    # Tilpasserer stien til loggfiler og JSON-registerdata etter behov
    run_full_workflow(
        logs_dir="2026",
        registers_json_path="store_config.json",
        output_dir="output",
    )


Ingen loggfiler funnet i den angitte mappen.


Outlier-rensingen gjøres før valutakorrigeringen slik at NOK-prisene ikke selv blir oppfattet som outliers, men en svakhet er at rapportene summerer priser i ulike valutaer uten omregning, slik at omsetningstallene ikke er i en felles valuta. 

In [31]:
out = Path("output")
display(pd.read_csv(out / "cleaned_transactions.csv").head())
display(pd.read_csv(out / "status_pivot_table.csv"))
display(pd.read_csv(out / "item_refund_ratios.csv"))
print((out / "hourly_revenue.json").read_text()[:500])

,timestamp,register_id,item,price,currency,status
0,2026-10-15 07:00:16,POS_01,Latte,4.86,EUR,REFUND
1,2026-10-15 07:00:18,POS_01,Cappuccino,4.44,USD,SUCCESS
2,2026-10-15 07:00:18,POS_02,Muffin,3.89,EUR,REFUND
3,2026-10-15 07:00:40,POS_02,Muffin,3.99,EUR,SUCCESS
4,2026-10-15 07:01:04,POS_02,Latte,49.40,NOK,SUCCESS


,item,CANCELLED,REFUND,SUCCESS
0,Cappuccino,476.06,216.93,1128.42
1,Croissant,319.02,307.74,1266.29
2,Espresso,281.64,283.62,1009.45
3,Latte,426.31,377.48,1798.76
4,Muffin,318.55,273.45,1094.67
5,Sandwich,722.96,598.72,2453.25


,item,refund_ratio
0,Cappuccino,0.253731
1,Croissant,0.182927
2,Espresso,0.215190
3,Latte,0.212766
4,Muffin,0.298701
5,Sandwich,0.240506


[
  {
    "item":"Cappuccino",
    "timestamp":"2026-10-15T07:00:00.000",
    "price":1033.3
  },
  {
    "item":"Cappuccino",
    "timestamp":"2026-10-15T08:00:00.000",
    "price":763.04
  },
  {
    "item":"Cappuccino",
    "timestamp":"2026-10-15T09:00:00.000",
    "price":25.07
  },
  {
    "item":"Croissant",
    "timestamp":"2026-10-15T07:00:00.000",
    "price":785.72
  },
  {
    "item":"Croissant",
    "timestamp":"2026-10-15T08:00:00.000",
    "price":961.77
  },
  {
    "item":"Crois


Over kan du se resultatene vi har fått

## Diskusjon
""
- **Rekursjon:** vi forsto ikke først hvordan resultatene fra hvert rekursive kall ble slått sammen. Vi trodde det skyldtes at listene hadde samme navn, men lærte at hvert kall har sin egen lokale variabel og at det er return og extend som kobler dem sammen.
- **Rekursiv JSON-traversering (2.2):** vi visste ikke hvordan vi skulle gå gjennom en struktur uten å kjenne oppbygningen. Det løste seg da vi tenkte på det som tre tilfeller: dict, liste og alt annet.
- **NumPy og Pandas (3 og 4):** vi var usikre på hvordan outlier-rensingen og refusjonsforholdet skulle implementeres, og blandet sammen hvilke deloppgaver som brukte hverandre.
- **Praktiske problemer:** notebooken lå i feil mappe, slik at stiene ikke stemte, og vi fikk advarsler fordi vi både definerte og importerte de samme funksjonene.
"""